# Data exploration

Agriculture-Vision 2021 train+val as the per-view stats table sees it (`data/stats.npz`, built by `python -m farmvlm.stats` on Colab). Outputs stay uncommitted: tiles must never be posted.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from farmvlm.stats import StatsTable
from farmvlm.folds import make_folds, tile_presence
from farmvlm.items import findings_of
from farmvlm.scheme import CLASSES, VIEWS, EXTENT_CUTS
t = StatsTable.load('../data/stats.npz')
print(len(t.tile), 'tiles,', len(t), 'views,', len(np.unique(t.field)), 'fields')

In [ ]:
pres = tile_presence(t)
for i, c in sorted(enumerate(CLASSES), key=lambda x: -pres[:, x[0]].sum()):
    print(f'{c.name:20s} {pres[:, i].sum():6d} tiles  {pres[:, i].mean():.1%}')
print('tiles with 2+ classes:', (pres.sum(1) >= 2).mean())

In [ ]:
full = t.row_view == VIEWS.index('full')
area = t.area[full][t.area[full] > 0]
plt.hist(area, bins=60); [plt.axvline(c, color='k', ls='--') for c in EXTENT_CUTS]
plt.xlabel('share of in-field area'); plt.ylabel('class instances'); plt.title('Extent cut points at the rounded tertiles'); plt.show()

In [ ]:
fs = [f for r in np.flatnonzero(full)[:20000] for f in findings_of(t.area[r], t.cells[r])]
print({k: round(np.mean(v), 3) for k, v in {'tiny': [f.tiny for f in fs], 'extent borderline': [' to ' in f.extent for f in fs], 'location borderline': [' or ' in f.loc for f in fs]}.items()})

In [ ]:
fold = make_folds(t, seed=0)
print('tiles per fold:', np.bincount(fold))
splits = {}
for f, s in zip(t.field, t.split): splits.setdefault(f, set()).add(s)
print('fields in both official train and val:', sum(len(v) > 1 for v in splits.values()), 'of', len(splits))